# 12.1 工具使用 (Tool Use)

工具使用是Agent的核心能力，使模型能调用外部工具完成自身无法完成的任务。

本节涵盖：函数调用、工具学习、API调用

## 1. 函数调用

**目的**：让模型生成结构化的函数调用指令

**基本原理**：定义工具的名称、描述和参数schema，模型根据用户需求生成函数调用JSON，执行引擎解析并调用对应函数。

**核心流程**：
1. 定义工具schema（名称、描述、参数类型）
2. 将工具描述注入提示
3. 模型生成函数调用JSON
4. 执行函数并返回结果
5. 模型基于结果继续生成

In [ ]:
import torch
import torch.nn as nn
import json

torch.manual_seed(42)

tools = [
    {'name': 'get_weather', 'description': 'Get weather for a location',
     'parameters': {'location': 'string', 'unit': 'string'}},
    {'name': 'search', 'description': 'Search the web',
     'parameters': {'query': 'string'}},
    {'name': 'calculator', 'description': 'Calculate math expression',
     'parameters': {'expression': 'string'}},
]

class ToolCallGenerator(nn.Module):
    def __init__(self, d=64, n_tools=3):
        super().__init__()
        self.tool_classifier = nn.Linear(d, n_tools)
        self.arg_generator = nn.Linear(d, 32)

    def forward(self, query_embed):
        tool_logits = self.tool_classifier(query_embed)
        tool_probs = torch.softmax(tool_logits, dim=-1)
        args = self.arg_generator(query_embed)
        return tool_probs, args

generator = ToolCallGenerator()
query = torch.randn(1, 64)
tool_probs, args = generator(query)

selected_tool = tool_probs.argmax(dim=-1).item()
tool_confidence = tool_probs.max().item()

print('=== Function Calling ===')
print(f'Available tools: {[t["name"] for t in tools]}')
print(f'Selected tool: {tools[selected_tool]["name"]} (confidence: {tool_confidence:.2f})')
print(f'Tool probabilities: {tool_probs[0].detach().tolist()}')
print(f'\nGenerated function call:')
call = {'name': tools[selected_tool]['name'], 'arguments': {'param': 'value'}}
print(f'  {json.dumps(call, indent=2)}')
print(f'\nKey: Model generates structured function calls from natural language.')

## 2. 工具学习与API调用

**工具学习**：模型通过训练学习何时、如何使用工具，包括工具选择和参数生成。

**API调用**：将外部API封装为工具，模型生成API调用请求，系统执行并返回结果。

**关键挑战**：
- 工具选择：从多个工具中选择最合适的
- 参数生成：生成正确的参数格式
- 错误处理：处理API调用失败的情况
- 多步调用：组合多个工具完成复杂任务

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class MultiToolAgent(nn.Module):
    def __init__(self, d=64, n_tools=5, max_steps=3):
        super().__init__()
        self.step_encoder = nn.GRU(d, 32, batch_first=True)
        self.tool_selector = nn.Linear(32, n_tools)
        self.stop_classifier = nn.Linear(32, 2)
        self.max_steps = max_steps

    def forward(self, query, prev_results=None):
        if prev_results is None:
            prev_results = query.unsqueeze(1)
        else:
            prev_results = torch.cat([prev_results, query.unsqueeze(1)], dim=1)
        _, h = self.step_encoder(prev_results)
        h = h.squeeze(0)
        tool_probs = torch.softmax(self.tool_selector(h), dim=-1)
        stop_probs = torch.softmax(self.stop_classifier(h), dim=-1)
        return tool_probs, stop_probs

agent = MultiToolAgent()
query = torch.randn(1, 64)
results = query.unsqueeze(1)

print('=== Multi-Step Tool Use ===')
for step in range(3):
    tool_probs, stop_probs = agent(query, results)
    tool = tool_probs.argmax(dim=-1).item()
    should_stop = stop_probs.argmax(dim=-1).item()
    print(f'Step {step+1}: tool={tool}, stop={should_stop}, confidence={tool_probs.max():.3f}')
    if should_stop:
        print(f'  -> Agent decided to stop')
        break
    result = torch.randn(1, 64)
    results = torch.cat([results, result.unsqueeze(1)], dim=1)

print(f'\nKey: Agents can chain multiple tool calls to complete complex tasks.')
print(f'Stop condition prevents infinite tool calling loops.')